In [25]:
import os
import os.path as osp
import random
from argparse import ArgumentParser
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm


from dataloader import Loaders
from models import Transformer

In [26]:
import altair as alt

[attention visualization ref](https://nlp.seas.harvard.edu/annotated-transformer/#results)

In [27]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if use multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [28]:
data_dir='/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/transformer/data.csv'
model_dir='/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/transformer/pths'
device="cuda" if torch.cuda.is_available() else "cpu"
num_workers=4
batch_size=8
val_num_workers=4
val_batch_size=1
max_token_length=512
q_dim=512
len_vocab=64101
start_idx=64100
end_idx=1
padding_idx=0
unk_idx=2
label_smoothing=0.1
learning_rate=1
warmup_steps=4000
max_epoch=1
val_interval=1
save_interval=1
resume_name='transformer_koren_20251115_123707_latest'
seed=42
wandb_mode='disabled'
wandb_run_name='KorEnTransformer'
debug=False

In [34]:
import warnings

# UserWarning 중에서 메시지에 'Automatically deduplicated selection parameter'가 포함된 경고 무시
warnings.filterwarnings("ignore", message="Automatically deduplicated selection parameter", category=UserWarning)


In [ ]:
def check_outputs(
    model,
    loaders,
    n_examples=15,
):
    results = [()] * n_examples
    for idx in range(n_examples):
        print("\nExample %d ========\n" % idx)
        batch = next(iter(loaders.loader_val))
        inputs = batch['input_ids'].to(device)
        # (1, src_seq_len)

        # teacher forcing에 사용할 gts
        # gts = batch_test['decoder_inputs'].to(device)
        # (batch_size, tgt_seq_len)
        # @@@ test 과정에서는 필요 없음

        # loss 계산에 사용할 정답 레이블
        labels = batch['labels'].to(device)
        # (1, tgt_seq_len)

        batch_test_ntokens = batch['ntokens']

        x_masks = batch['attention_mask'].to(device)

        # gt_masks = batch_test['decoder_mask'].to(device)
        # @@@ test 과정에서는 필요 없음

        preds = model.inference(inputs, x_masks, labels.size(-1) * 2)
        # (1, pred_seq_len)


        # @@@@ 수정 필요
        # @@@@ 한 문장의 토큰들을 다 합친 결과가 아니라 토큰들이 각 원소로 들어가있어야 함
        decoded_inputs = loaders.tokenizer.convert_ids_to_tokens(inputs[0])

        decoded_preds = loaders.tokenizer.convert_ids_to_tokens(preds[0])

        decoded_labels = loaders.tokenizer.convert_ids_to_tokens(labels[0])

        # Some simple post-processing
        decoded_inputs = [inp.strip() for inp in decoded_inputs if inp != "<pad>"]
        decoded_preds = [pred.strip() for pred in decoded_preds if pred != "<pad>"]
        decoded_labels = [label.strip() for label in decoded_labels if label != "<pad>"]
        # batch size가 1이면 loader의 collate_fn 설정에 의해 pad 토큰이 없지만 일단 유지

        print(
            "Source Text (Input)        : "
            + " ".join(decoded_inputs)
        )
        print(
            "Target Text (Ground Truth) : "
            + " ".join(decoded_labels)
        )
        print("Model Output             : " 
              + " ".join(decoded_preds)
        )
        results[idx] = (batch, decoded_inputs, decoded_labels, preds, decoded_preds)
    return results


def run_model_example(n_examples=5):

    print("Preparing Data ...")
    loaders = Loaders(data_path=data_dir, max_token_length=max_token_length, batch_size_train=batch_size, num_workers=num_workers, batch_size_val=val_batch_size, batch_size_test=val_batch_size, val_num_workers=val_num_workers, start_idx=start_idx, end_idx=end_idx, padding_idx=padding_idx, unk_idx=unk_idx, seed=seed)

    print("Loading Trained Model ...")

    # Initialize the model
    model = Transformer(
        src_len_vocab=len_vocab, 
        tgt_len_vocab=len_vocab, 
        start_idx=start_idx, 
        end_idx=end_idx, 
        padding_idx=padding_idx, 
        unk_idx=unk_idx, 
        q_dim=q_dim, 
        k_dim=q_dim, 
        v_dim=q_dim, 
        h_dim=(q_dim * 4), 
        visualization=True
    )
    # KETI-AIR/ke-t5-base tokenizer의 한영 통합 토큰 종류 수는 64100 + 시작 토큰 1개 추가해서 = 64101개

    load_dict = None

    load_dict = torch.load(
        osp.join(model_dir, f"{resume_name}.pth"), map_location="cpu"
    )
    model.load_state_dict(load_dict["model_state_dict"])

    model.to(device)

    # 모델을 검증모드로 변경
    model.eval()

    print("Checking Model Outputs:")
    example_data = check_outputs(
        model, loaders, n_examples=n_examples
    )
    return model, example_data



In [31]:
def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    "convert a dense matrix to a data frame with row and column indices"
    return pd.DataFrame(
        [
            (
                r,
                c,
                float(m[r, c]),
                "%.3d %s"
                % (r, row_tokens[r] if len(row_tokens) > r else "<blank>"),
                "%.3d %s"
                % (c, col_tokens[c] if len(col_tokens) > c else "<blank>"),
            )
            for r in range(m.shape[0])
            for c in range(m.shape[1])
            if r < max_row and c < max_col
        ],
        # if float(m[r,c]) != 0 and r < max_row and c < max_col],
        columns=["row", "column", "value", "row_token", "col_token"],
    )


def attn_map(attn, layer, head, row_tokens, col_tokens, max_dim=30):
    df = mtx2df(
        attn[0, head].data, # attn은 (batch, head, q_seq_len, k_seq_len) ==> 지정된 head의 (q_seq_len, k_seq_len) 부분만 추출
        max_dim,
        max_dim,
        row_tokens,
        col_tokens,
    )

    # param = alt.selection_point(name=f'head{head}')

    return (
        alt.Chart(data=df)
        .mark_rect()
        .encode(
            x=alt.X("col_token", axis=alt.Axis(title="")),
            y=alt.Y("row_token", axis=alt.Axis(title="")),
            color="value",
            tooltip=["row", "column", "value", "row_token", "col_token"],
        )
        .properties(height=400, width=400)
        .interactive()
        # .add_params(param)
    )

In [32]:
def get_encoder(model, layer):
    return model.encoder.blocks[layer].MHA.attn


def get_decoder_self(model, layer):
    return model.decoder.blocks[layer].MMHA.attn


def get_decoder_src(model, layer):
    return model.decoder.blocks[layer].MHA.attn


def visualize_layer(model, layer, getter_fn, ntokens, row_tokens, col_tokens):
    # ntokens = last_example[0].ntokens
    attn = getter_fn(model, layer)
    n_heads = attn.shape[1]

    charts = [
        attn_map(
            attn,
            0,
            h,
            row_tokens=row_tokens,
            col_tokens=col_tokens,
            max_dim=ntokens,
        )
        for h in range(n_heads)
    ]
    assert n_heads == 8

    combined_chart = alt.vconcat(
        charts[0]
        # | charts[1]
        | charts[2]
        # | charts[3]
        | charts[4]
        # | charts[5]
        | charts[6]
        # | charts[7]
        # layer + 1 due to 0-indexing
    ).properties(title="Layer %d" % (layer + 1))


    return combined_chart

In [35]:
def viz_encoder_self():
    model, example_data = run_model_example(n_examples=1)
    # example_data는 [(batch, decoded_inputs, decoded_labels, preds, decoded_preds), ...]
    example = example_data[
        len(example_data) - 1
    ]  # batch object for the final example


    # selections = [alt.selection_point(name=f'head{h}') for h in range(8)]

    layer_viz = [
        visualize_layer(
            model, layer, get_encoder, len(example[1]), example[1], example[1] # example[1]은 decoded_inputs ==> len(example[1])로 input의 토큰 개수 입력
        )
        for layer in range(6)
    ]

    combined_chart = alt.hconcat(
        layer_viz[0]
        # & layer_viz[1]
        & layer_viz[2]
        # & layer_viz[3]
        & layer_viz[4]
        # & layer_viz[5]
    )


    # # 상위 차트에만 selection 추가
    # for i, sel in enumerate(selections):
    #     if i in [0, 2, 4, 6]:
    #         combined_chart = combined_chart.add_params(sel)


    return combined_chart

viz_encoder_self()

Preparing Data ...
==>> unique_classes: [2, 4, 1, 0, 3, 5]
==>> self.tokenizer.model_max_length: 512
==>> len(self.tokenizer): 64101
==>> self.datasets: DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1281934
    })
    validation: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 288435
    })
    test: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 32049
    })
})


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading Trained Model ...
Checking Model Outputs:

Example 0 ========

Source Text (Input)        : ▁이 ▁쿠키 가 ▁이 ▁상자 ▁안에 ▁들어갈 까요 ? </s>
Target Text (Ground Truth) : ▁Would ▁this ▁cookie ▁be ▁fit ▁in ▁the ▁box ? </s>
Model Output             : ▁Would ▁this ▁ s ex y ▁in ▁the ▁box ? </s>


alt.HConcatChart(...)

In [36]:
def viz_decoder_self():
    model, example_data = run_model_example(n_examples=1)
    # example_data는 [(batch, decoded_inputs, decoded_labels, preds, decoded_preds), ...]
    example = example_data[len(example_data) - 1]

    layer_viz = [
        visualize_layer(
            model,
            layer,
            get_decoder_self,
            len(example[4]),
            example[4],
            example[4],
        )
        for layer in range(6)
    ]
    return alt.hconcat(
        layer_viz[0]
        & layer_viz[1]
        & layer_viz[2]
        & layer_viz[3]
        & layer_viz[4]
        & layer_viz[5]
    )


viz_decoder_self()

Preparing Data ...
==>> unique_classes: [2, 4, 1, 0, 3, 5]
==>> self.tokenizer.model_max_length: 512
==>> len(self.tokenizer): 64101
==>> self.datasets: DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1281934
    })
    validation: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 288435
    })
    test: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 32049
    })
})


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading Trained Model ...
Checking Model Outputs:

Example 0 ========

Source Text (Input)        : ▁이 ▁쿠키 가 ▁이 ▁상자 ▁안에 ▁들어갈 까요 ? </s>
Target Text (Ground Truth) : ▁Would ▁this ▁cookie ▁be ▁fit ▁in ▁the ▁box ? </s>
Model Output             : ▁Is ▁this ▁cave ▁open ▁in ▁this ▁box ? </s>


alt.HConcatChart(...)

In [39]:
def viz_decoder_src():
    model, example_data = run_model_example(n_examples=1)
    example = example_data[len(example_data) - 1]

    print(f"==>> len(example[1]): {len(example[1])}")
    print(f"==>> len(example[4]): {len(example[4])}")
    print(f"==>> max(len(example[1]), len(example[4])): {max(len(example[1]), len(example[4]))}")

    layer_viz = [
        visualize_layer(
            model,
            layer,
            get_decoder_src,
            max(len(example[1]), len(example[4])),
            example[1],
            example[4],
        )
        for layer in range(6)
    ]
    return alt.hconcat(
        layer_viz[0]
        & layer_viz[1]
        & layer_viz[2]
        & layer_viz[3]
        & layer_viz[4]
        & layer_viz[5]
    )


viz_decoder_src()

Preparing Data ...
==>> unique_classes: [2, 4, 1, 0, 3, 5]
==>> self.tokenizer.model_max_length: 512
==>> len(self.tokenizer): 64101
==>> self.datasets: DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1281934
    })
    validation: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 288435
    })
    test: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 32049
    })
})


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/paokimsiwoong/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading Trained Model ...
Checking Model Outputs:

Example 0 ========

Source Text (Input)        : ▁이 ▁쿠키 가 ▁이 ▁상자 ▁안에 ▁들어갈 까요 ? </s>
Target Text (Ground Truth) : ▁Would ▁this ▁cookie ▁be ▁fit ▁in ▁the ▁box ? </s>
Model Output             : ▁Is ▁this ▁ice ing ▁open ▁in ▁this ▁box ? </s>
==>> len(example[1]): 10
==>> len(example[4]): 10
==>> max(len(example[1]), len(example[4])): 10


alt.HConcatChart(...)